# 00 — SCD Type 0: Retain Original

Schema:

```text
existing key changed -> ignore
new key              -> insert
```

In [1]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import *

spark = SparkSession.getActiveSession()
if spark is not None:
    spark.stop()

spark = (
    SparkSession.builder
    .appName("scd_type_0_retain_original")
    .master("spark://spark-master:7077")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
spark.version

'4.0.2'

In [2]:
current_schema = StructType([
    StructField("user_id", StringType(), False),
    StructField("signup_date_raw", StringType(), False),
    StructField("country", StringType(), False),
])

current = (
    spark.createDataFrame(
        [("u1", "2026-01-01", "SK"), ("u2", "2026-01-02", "CZ")],
        current_schema,
    )
    .withColumn("signup_date", F.to_date("signup_date_raw"))
    .drop("signup_date_raw")
)

incoming = (
    spark.createDataFrame(
        [("u1", "2026-02-01", "AT"), ("u3", "2026-02-03", "DE")],
        current_schema,
    )
    .withColumn("signup_date", F.to_date("signup_date_raw"))
    .drop("signup_date_raw")
)

current.show(truncate=False)
incoming.show(truncate=False)

+-------+-------+-----------+
|user_id|country|signup_date|
+-------+-------+-----------+
|u1     |SK     |2026-01-01 |
|u2     |CZ     |2026-01-02 |
+-------+-------+-----------+

+-------+-------+-----------+
|user_id|country|signup_date|
+-------+-------+-----------+
|u1     |AT     |2026-02-01 |
|u3     |DE     |2026-02-03 |
+-------+-------+-----------+



In [3]:
new_keys_only = incoming.join(current.select("user_id"), on="user_id", how="left_anti")
scd0_result = current.unionByName(new_keys_only).orderBy("user_id")

new_keys_only.show(truncate=False)
scd0_result.show(truncate=False)

+-------+-------+-----------+
|user_id|country|signup_date|
+-------+-------+-----------+
|u3     |DE     |2026-02-03 |
+-------+-------+-----------+

+-------+-------+-----------+
|user_id|country|signup_date|
+-------+-------+-----------+
|u1     |SK     |2026-01-01 |
|u2     |CZ     |2026-01-02 |
|u3     |DE     |2026-02-03 |
+-------+-------+-----------+



In [4]:
totals = spark.createDataFrame(
    [
        ("current_rows", current.count()),
        ("incoming_rows", incoming.count()),
        ("inserted_new_keys", new_keys_only.count()),
        ("final_rows", scd0_result.count()),
    ],
    StructType([StructField("metric", StringType(), False), StructField("value", LongType(), False)])
)
totals.show(truncate=False)

+-----------------+-----+
|metric           |value|
+-----------------+-----+
|current_rows     |2    |
|incoming_rows    |2    |
|inserted_new_keys|1    |
|final_rows       |3    |
+-----------------+-----+



In [5]:
spark.stop()